# Building a GPT from Scratch

The Transformer is not magic. It is a sequence of matrix multiplications, normalizations, and nonlinearities — every one of which can be derived, implemented, and inspected. This notebook builds a complete GPT-style language model from scratch in pure PyTorch, deriving each component from first principles: scaled dot-product attention, causal masking, multi-head attention, RoPE positional encoding, residual connections, RMSNorm, and SwiGLU FFN. The result is a ~200-line implementation that we will use and instrument throughout the rest of the series.

:::{.callout-note}
## References
The architecture here closely follows [nanoGPT](https://github.com/karpathy/nanoGPT) — specifically [`model.py`](https://github.com/karpathy/nanoGPT/blob/master/model.py) (`CausalSelfAttention`, `Block`, `GPT`). We modernize two components relative to GPT-2: RMSNorm in place of LayerNorm, and SwiGLU in place of the GELU FFN, matching the LLaMA/DeepSeek family.

:::

## The Problem Attention Solves

A language model needs to produce a representation of each token that is *context-dependent* — the word "bank" means something different in "river bank" and "bank account". Fixed positional encodings or simple averaging destroy the relational structure. What we want is: for each token position $i$, produce an output that is an [*informed weighted combination*]{.underline} of all other tokens, where the weights are computed from the content of the tokens themselves.

That is [exactly what attention computes]{.mark}.

## Scaled Dot-Product Attention

Start with a sequence of $n$ token embeddings packed into a matrix $X \in \mathbb{R}^{n \times d}$. We project this into three matrices using learned weight matrices $W_Q, W_K, W_V \in \mathbb{R}^{d \times d_k}$:

$$Q = X W_Q, \quad K = X W_K, \quad V = X W_V$$

$Q$ (queries), $K$ (keys), and $V$ (values) are all $\in \mathbb{R}^{n \times d_k}$. The attention output is:

$$\text{Attention}(Q, K, V) = \text{softmax}\!\left(\frac{Q K^\top}{\sqrt{d_k}}\right) V$$

**$QK^\top$** is an $n \times n$ matrix of dot products. Entry $(i, j)$ measures how much query $i$ attends to key $j$ — a similarity score between position $i$'s query vector and position $j$'s key vector.

**The $\sqrt{d_k}$ scale factor.** [This is critical]{.underline}. If $q$ and $k$ are random vectors with zero mean and unit variance, then $q \cdot k = \sum_{l=1}^{d_k} q_l k_l$ has variance $d_k$. Its standard deviation is therefore $\sqrt{d_k}$. Dividing by $\sqrt{d_k}$ brings the dot products back to unit variance. Without this, large $d_k$ pushes the softmax inputs into regions with extremely small gradients — the softmax saturates, one attention weight dominates, and the network learns slowly or not at all.

**$\text{softmax}(\cdot)$** converts the score matrix into a row-stochastic matrix of attention weights $A \in \mathbb{R}^{n \times n}$, where each row sums to 1. Entry $A_{ij}$ is the weight position $i$ assigns to position $j$.

**$AV$** is a weighted combination of value vectors. Output row $i$ is $\sum_j A_{ij} v_j$ — a mixture of all value vectors weighted by how much position $i$ attends to each position.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
from dataclasses import dataclass


def scaled_dot_product_attention(Q, K, V, mask=None):
    """
    Q: (batch, n_heads, seq_len, d_k)
    K: (batch, n_heads, seq_len, d_k)
    V: (batch, n_heads, seq_len, d_k)
    mask: (1, 1, seq_len, seq_len) — True where we want to mask (set to -inf)
    """
    d_k = Q.size(-1)
    scores = Q @ K.transpose(-2, -1) / math.sqrt(d_k)     # (B, H, T, T)
    if mask is not None:
        scores = scores.masked_fill(mask, float('-inf'))
    weights = F.softmax(scores, dim=-1)
    return weights @ V, weights

## Causal Masking

For a language model, position $i$ should [only attend to positions $\leq i$]{.mark} — it cannot see future tokens. We enforce this by masking the upper triangle of the attention score matrix to $-\infty$ before the softmax. After softmax, $e^{-\infty} = 0$, so those positions receive zero attention weight.

In [ ]:
def make_causal_mask(seq_len, device):
    # Upper triangle (excluding diagonal) → True → masked to -inf
    # Shape: (1, 1, seq_len, seq_len) — broadcasts over batch and heads
    mask = torch.triu(
        torch.ones(seq_len, seq_len, dtype=torch.bool, device=device),
        diagonal=1
    )
    return mask.unsqueeze(0).unsqueeze(0)

mask = make_causal_mask(4, device='cpu').squeeze()
print(mask.int())

After `masked_fill`, the score matrix looks like:

```
position:  0    1    2    3
        0 [s00  -∞   -∞   -∞ ]   position 0 only attends to itself
        1 [s10  s11  -∞   -∞ ]   position 1 attends to 0 and 1
        2 [s20  s21  s22  -∞ ]
        3 [s30  s31  s32  s33]   position 3 attends to all prior positions
```

## Multi-Head Attention

A single attention head computes one weighted combination of values — one perspective on the sequence. **Multi-head attention** computes $h$ such perspectives in parallel, each with its own learned projections, then concatenates and re-projects:

$$\text{MultiHead}(X) = \text{Concat}(\text{head}_1, \ldots, \text{head}_h) W_O$$

where $\text{head}_i = \text{Attention}(X W_Q^{(i)}, X W_K^{(i)}, X W_V^{(i)})$ and $W_O \in \mathbb{R}^{h d_k \times d}$.

**Why multiple heads?** A single head is forced to mix all relational patterns into one weighted average. Multiple heads can [*specialize*]{.underline}: one head might learn syntactic dependencies, another coreference, another local context.

**The efficiency trick.** Rather than maintaining $h$ separate weight matrices, we project $X$ to $\mathbb{R}^{n \times d}$ once (with $d = h \cdot d_k$), then reshape to split across heads. This is equivalent to $h$ separate projections but runs as a single batched matrix multiply.

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model: int, n_heads: int):
        super().__init__()
        assert d_model % n_heads == 0
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_k     = d_model // n_heads

        self.W_qkv = nn.Linear(d_model, 3 * d_model, bias=False)  # <1>
        self.W_o   = nn.Linear(d_model, d_model,     bias=False)

    def forward(self, x, rope, mask=None):
        B, T, C = x.shape

        qkv = self.W_qkv(x)                       # (B, T, 3*d_model)
        Q, K, V = qkv.split(self.d_model, dim=-1)

        def split_heads(t):
            return t.view(B, T, self.n_heads, self.d_k).transpose(1, 2)

        Q, K, V = split_heads(Q), split_heads(K), split_heads(V)  # (B, H, T, d_k)
        Q, K = rope(Q), rope(K)                   # <2>

        attn_out, _ = scaled_dot_product_attention(Q, K, V, mask)
        attn_out = attn_out.transpose(1, 2).contiguous().view(B, T, C)  # <3>
        return self.W_o(attn_out)

1. A single projection matrix for Q, K, V — we split the output after the matmul rather than maintaining three separate weight matrices.
2. RoPE is applied to Q and K only — V is not rotated since it does not participate in the dot product.
3. The `.contiguous()` call before `.view()` is required because `.transpose()` produces a non-contiguous tensor. `.view()` requires contiguous memory. [This is a common source of bugs]{.mark} — an alternative is `.reshape()`, which handles non-contiguous tensors but may copy.

## Rotary Positional Encoding (RoPE)

Attention is *permutation-equivariant*: if you shuffle the input sequence, the output shuffles in the same way. Nothing in the attention mechanism itself knows that token 3 comes after token 2. We need to [inject positional information]{.underline}.

The original Transformer used fixed sinusoidal encodings added to the embeddings. Modern LLMs use **Rotary Positional Encoding (RoPE)**, which encodes position by *rotating* the query and key vectors before the dot product. The key property: for query at position $m$ and key at position $n$:

$$\text{RoPE}(q, m) \cdot \text{RoPE}(k, n) = f(q, k, m - n)$$

The dot product depends [only on the *relative* offset $m - n$]{.mark}, not the absolute positions $m$ and $n$ separately. This is why RoPE generalizes better to sequence lengths longer than seen during training.

### The rotation

For a $d_k$-dimensional vector, split it into $d_k/2$ pairs and rotate each pair by a different angle. For position $m$, pair $l$ is rotated by $m \cdot \theta_l$ where $\theta_l = 10000^{-2l/d_k}$. Consider a 2D pair $[x_1, x_2]$ as a complex number $z = x_1 + i x_2$. Multiplying by $e^{i m\theta_l}$ rotates $z$ by $m\theta_l$:

$$z' = z \cdot e^{i m\theta_l} = (x_1 \cos m\theta_l - x_2 \sin m\theta_l) + i(x_1 \sin m\theta_l + x_2 \cos m\theta_l)$$

In practice we apply this rotation directly to real vectors without complex arithmetic.

In [ ]:
def rotate_half(x):
    """Rotate pairs: [..., x1, x2, ...] → [..., -x2, x1, ...]"""
    x1 = x[..., : x.shape[-1] // 2]
    x2 = x[..., x.shape[-1] // 2 :]
    return torch.cat([-x2, x1], dim=-1)


class RoPE(nn.Module):
    def __init__(self, d_k: int, max_seq_len: int = 2048, base: int = 10000):
        super().__init__()
        # θ_l = 1 / 10000^(2l/d_k),  shape: (d_k/2,)
        inv_freq = 1.0 / (base ** (torch.arange(0, d_k, 2).float() / d_k))
        self.register_buffer('inv_freq', inv_freq)
        self._build_cache(max_seq_len)

    def _build_cache(self, seq_len: int):
        t = torch.arange(seq_len, device=self.inv_freq.device).float()
        freqs = torch.outer(t, self.inv_freq)       # (seq_len, d_k/2)
        emb   = torch.cat([freqs, freqs], dim=-1)   # (seq_len, d_k)
        self.register_buffer('cos_cached', emb.cos()[None, None, :, :])
        self.register_buffer('sin_cached', emb.sin()[None, None, :, :])

    def forward(self, x):
        """
        x: (batch, n_heads, seq_len, d_k)
        Returns x with RoPE applied.
        """
        seq_len = x.size(2)
        cos = self.cos_cached[:, :, :seq_len, :]
        sin = self.sin_cached[:, :, :seq_len, :]
        return (x * cos) + (rotate_half(x) * sin)

## Residual Connections

### The problem: gradient attenuation in deep networks

Consider a network of $L$ layers with no residual connections:

$$y = f_L(f_{L-1}(\cdots f_1(x) \cdots))$$

The gradient of the loss $\mathcal{L}$ with respect to the input of layer $l$ is, by the chain rule:

$$\frac{\partial \mathcal{L}}{\partial x_l} = \frac{\partial \mathcal{L}}{\partial x_L} \cdot \prod_{i=l}^{L-1} \frac{\partial f_{i+1}}{\partial x_{i+1}}$$

This is a product of $L - l$ Jacobians. If each has spectral norm less than 1 — which happens when weights are small or activations are saturating — this product shrinks exponentially with depth. The gradient reaching layer $l$ is $O(\lambda^{L-l})$ for some $\lambda < 1$: the **vanishing gradient problem**.

In [ ]:
torch.manual_seed(0)

class DeepNetNoResidual(nn.Module):
    def __init__(self, depth=12, d=64):
        super().__init__()
        self.layers = nn.ModuleList([
            nn.Sequential(nn.Linear(d, d), nn.Tanh())
            for _ in range(depth)
        ])
    def forward(self, x):
        for layer in self.layers:
            x = layer(x)
        return x

model = DeepNetNoResidual(depth=12, d=64)
x = torch.randn(8, 64)
model(x).mean().backward()

for i, layer in enumerate(model.layers):
    g = layer[0].weight.grad.norm().item()
    print(f"layer {i:2d}  grad_norm={g:.2e}")

Gradients shrink by roughly an order of magnitude every few layers going backwards. [Early layers are effectively frozen]{.underline} — they will not learn.

### The residual fix

A **residual block** wraps a sublayer $F$ with a skip connection:

$$y = x + F(x)$$

The gradient through this block is:

$$\frac{\partial \mathcal{L}}{\partial x} = \frac{\partial \mathcal{L}}{\partial y} \cdot \left(I + \frac{\partial F}{\partial x}\right)$$

The **identity matrix $I$ is the critical term**. Even if $\frac{\partial F}{\partial x} \approx 0$ — the sublayer is saturated or poorly initialized — the gradient still flows through the skip connection unchanged:

$$\frac{\partial \mathcal{L}}{\partial x} \approx \frac{\partial \mathcal{L}}{\partial y} \cdot I = \frac{\partial \mathcal{L}}{\partial y}$$

Over $L$ residual layers, there is [always a direct path]{.mark} from the loss to every layer through the chain of skip connections.

In [ ]:
class DeepNetWithResidual(nn.Module):
    def __init__(self, depth=12, d=64):
        super().__init__()
        self.layers = nn.ModuleList([
            nn.Sequential(nn.Linear(d, d), nn.Tanh())
            for _ in range(depth)
        ])
    def forward(self, x):
        for layer in self.layers:
            x = x + layer(x)
        return x

model = DeepNetWithResidual(depth=12, d=64)
x = torch.randn(8, 64)
model(x).mean().backward()

for i, layer in enumerate(model.layers):
    g = layer[0].weight.grad.norm().item()
    print(f"layer {i:2d}  grad_norm={g:.2e}")

Gradient norms are now roughly uniform across depth — every layer receives a usable gradient signal.

### The residual stream

In a Transformer, the residual connections create the **residual stream** — a $d_{\text{model}}$-dimensional vector that flows through every layer and accumulates information:

```
x_0 = token_embedding
x_1 = x_0 + Attention(RMSNorm(x_0))
x_2 = x_1 + FFN(RMSNorm(x_1))
x_3 = x_2 + Attention(RMSNorm(x_2))
...
```

Each sublayer reads from the stream, computes a delta, and adds it back. [The stream is never overwritten — only accumulated]{.mark}. This view makes it clear why the residual connection is not a mere engineering trick: it is the fundamental information-routing mechanism of the Transformer.[^stream]

[^stream]: The residual stream is a central concept in recent mechanistic interpretability work. Understanding it is key to understanding how information flows through Transformers.

## RMSNorm

Modern LLMs (LLaMA, DeepSeek, Mistral) use **RMSNorm** in place of LayerNorm. It normalizes by the root-mean-square of the activations, dropping the mean-centering step:

$$\text{RMSNorm}(x) = \gamma \cdot \frac{x}{\text{RMS}(x) + \epsilon}, \qquad \text{RMS}(x) = \sqrt{\frac{1}{d}\sum_{i=1}^d x_i^2}$$

This removes both the mean subtraction and the $\beta$ (bias) parameter relative to LayerNorm.

**Why?** LayerNorm's mean-centering is largely redundant for pre-norm Transformers — the sublayer inputs are already approximately zero-mean after initialization, and the normalization scale $\gamma$ handles the important variance control. RMSNorm is [slightly faster and simpler]{.mark}, with no empirical regression in quality.[^rmsnorm]

[^rmsnorm]: See Zhang & Sennrich (2019), "Root Mean Square Layer Normalization". At the scale of modern LLMs, the ~10% speed improvement from skipping mean centering is meaningful.

In [ ]:
class RMSNorm(nn.Module):
    def __init__(self, d_model: int, eps: float = 1e-6):
        super().__init__()
        self.eps   = eps
        self.gamma = nn.Parameter(torch.ones(d_model))

    def forward(self, x):
        # x: (batch, seq_len, d_model)
        rms = x.pow(2).mean(dim=-1, keepdim=True).add(self.eps).sqrt()
        return self.gamma * (x / rms)

## SwiGLU FFN

The original GPT FFN expands to $4d_{\text{model}}$, applies GELU, and projects back:

$$\text{FFN}_{\text{GELU}}(x) = W_2 \cdot \text{GELU}(W_1 x)$$

Modern LLMs (LLaMA, DeepSeek, PaLM) replace this with **SwiGLU**:

$$\text{FFN}_{\text{SwiGLU}}(x) = (W_1 x \odot \text{SiLU}(W_g x)) \, W_2$$

where $\text{SiLU}(z) = z \cdot \sigma(z)$ is the sigmoid linear unit (also called Swish) and $\odot$ is elementwise multiplication. There are now [three weight matrices]{.mark} instead of two: $W_1, W_g \in \mathbb{R}^{d \times d_{\text{ff}}}$ and $W_2 \in \mathbb{R}^{d_{\text{ff}} \times d}$.

**Why SwiGLU?** The gating mechanism ($W_1 x \odot \text{SiLU}(W_g x)$) acts as a learned content-dependent filter: $W_g x$ controls how much of the signal from $W_1 x$ passes through. Empirically, SwiGLU consistently outperforms GELU FFN at the same parameter budget.[^swiglu]

**Hidden dimension.** To keep parameter count comparable to a $4d$ GELU FFN with two matrices, SwiGLU uses $d_{\text{ff}} = \frac{2}{3} \cdot 4d = \frac{8d}{3}$. This compensates for the third weight matrix. In practice, $d_{\text{ff}}$ is rounded to a multiple of 64 or 256 for hardware efficiency.

[^swiglu]: See Noam Shazeer (2020), "GLU Variants Improve Transformer". SwiGLU is now the default FFN for most frontier models.

In [ ]:
class SwiGLU(nn.Module):
    def __init__(self, d_model: int, d_ff: int | None = None):
        super().__init__()
        if d_ff is None:
            # 2/3 * 4d rounded to nearest multiple of 64
            d_ff = int(2 * 4 * d_model / 3)
            d_ff = 64 * ((d_ff + 63) // 64)
        self.W_gate = nn.Linear(d_model, d_ff, bias=False)  # <1>
        self.W_up   = nn.Linear(d_model, d_ff, bias=False)
        self.W_down = nn.Linear(d_ff, d_model, bias=False)

    def forward(self, x):
        return self.W_down(F.silu(self.W_gate(x)) * self.W_up(x))  # <2>

1. `W_gate` produces the gating signal; `W_up` produces the content signal.
2. The elementwise product `F.silu(gate) * up` is the gated activation — the gate controls how much of each feature in `up` passes through to `W_down`.

## Weight Initialization

Standard Kaiming initialization is not quite right for Transformers. The problem is the residual stream: with $L$ residual blocks each adding a term of variance $\sigma^2$ to the stream, the stream variance after $L$ blocks is $L\sigma^2$ — it grows with depth.

GPT-2 addresses this by **scaling the output projections of each residual sublayer** (the final linear layer of both the attention $W_O$ and the FFN $W_{\text{down}}$) by $1/\sqrt{2L}$ at initialization, where $L$ is the number of Transformer layers:

$$\sigma_{\text{init}} \leftarrow \frac{0.02}{\sqrt{2L}}$$

With this scaling, each residual block contributes variance $\sigma^2 / (2L)$, and the total stream variance after $L$ blocks is $L \cdot \sigma^2/(2L) = \sigma^2/2$ — [bounded and independent of depth]{.mark}.

In [ ]:
def _init_weights(module: nn.Module, n_layers: int) -> None:
    if isinstance(module, nn.Linear):
        std = 0.02
        # Scale output projections of residual sublayers
        if getattr(module, '_is_residual_proj', False):   # <1>
            std /= math.sqrt(2 * n_layers)
        nn.init.normal_(module.weight, mean=0.0, std=std)
        if module.bias is not None:
            nn.init.zeros_(module.bias)
    elif isinstance(module, nn.Embedding):
        nn.init.normal_(module.weight, mean=0.0, std=0.02)

1. We tag output projection layers with a `_is_residual_proj` attribute when they are created in the GPT block, so the initializer can identify them without fragile name-matching.

## The GPT Block

We now have all the pieces. A single Transformer block reads from the residual stream, applies attention, adds back, applies FFN, adds back — all with pre-norm using RMSNorm:

In [ ]:
class TransformerBlock(nn.Module):
    def __init__(self, d_model: int, n_heads: int):
        super().__init__()
        self.norm1 = RMSNorm(d_model)
        self.attn  = MultiHeadAttention(d_model, n_heads)
        self.norm2 = RMSNorm(d_model)
        self.ffn   = SwiGLU(d_model)

        # Tag output projections for residual scaling at init
        self.attn.W_o._is_residual_proj  = True   # <1>
        self.ffn.W_down._is_residual_proj = True

    def forward(self, x, rope, mask=None):
        x = x + self.attn(self.norm1(x), rope, mask)   # <2>
        x = x + self.ffn(self.norm2(x))
        return x

1. Tagging output projections here, at construction time, so `_init_weights` can find them by attribute rather than by name substring matching.
2. Pre-norm: normalization is applied inside the residual branch. The skip connection path is completely clean — the gradient identity $(I + \partial F / \partial x)$ holds without any normalization in the way.

## The GPT Model

The full model: token embedding → Transformer blocks → final RMSNorm → LM head. RoPE is instantiated once and shared across all blocks — the rotation angles depend only on position and head dimension, not on learned parameters.

In [ ]:
@dataclass
class GPTConfig:
    vocab_size:  int = 50257
    d_model:     int = 768
    n_layers:    int = 12
    n_heads:     int = 12
    max_seq_len: int = 1024


class GPT(nn.Module):
    def __init__(self, config: GPTConfig):
        super().__init__()
        self.config = config

        self.token_embedding = nn.Embedding(config.vocab_size, config.d_model)
        self.rope   = RoPE(config.d_model // config.n_heads, config.max_seq_len)  # <1>
        self.blocks = nn.ModuleList([
            TransformerBlock(config.d_model, config.n_heads)
            for _ in range(config.n_layers)
        ])
        self.norm_out = RMSNorm(config.d_model)
        self.lm_head  = nn.Linear(config.d_model, config.vocab_size, bias=False)
        self.lm_head.weight = self.token_embedding.weight  # <2>

        self.apply(lambda m: _init_weights(m, config.n_layers))

    def forward(self, idx, targets=None):
        """
        idx:     (batch, seq_len) — token indices
        targets: (batch, seq_len) — next-token targets; if provided, also returns loss
        """
        B, T = idx.shape
        x    = self.token_embedding(idx)        # (B, T, d_model)
        mask = make_causal_mask(T, idx.device)

        for block in self.blocks:
            x = block(x, self.rope, mask)

        x      = self.norm_out(x)
        logits = self.lm_head(x)                # (B, T, vocab_size)

        loss = None
        if targets is not None:
            loss = F.cross_entropy(
                logits.view(-1, logits.size(-1)),
                targets.view(-1)
            )
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens: int, temperature: float = 1.0):
        """Autoregressive sampling — no KV cache (see NB13)."""
        for _ in range(max_new_tokens):
            idx_cond     = idx[:, -self.config.max_seq_len:]
            logits, _    = self(idx_cond)
            logits       = logits[:, -1, :] / temperature
            next_token   = torch.multinomial(F.softmax(logits, dim=-1), num_samples=1)
            idx          = torch.cat([idx, next_token], dim=1)
        return idx

1. RoPE is shared across all blocks. The rotation angles are purely a function of position and head dimension — they contain no learned parameters and need only be computed once.
2. **Weight tying:** the LM head shares its weight matrix with the token embedding. This reduces parameters and consistently improves perplexity. The intuition: the embedding maps tokens to vectors, and the LM head maps vectors back to token logits — they are inverses of each other, so sharing weights is geometrically natural.

## Parameter Count

Every parameter in the model is accounted for. Given config values $V$ (vocab size), $d$ (`d_model`), $n$ (`n_layers`), $h$ (`n_heads`), $d_{\text{ff}}$ (SwiGLU hidden dim $\approx 8d/3$ rounded up):

| Component | Parameters |
|---|---|
| Token embedding | $V \times d$ — shared with LM head |
| Per-block: $W_{QKV}$ | $3d^2$ |
| Per-block: $W_O$ | $d^2$ |
| Per-block: SwiGLU ($W_{\text{gate}}, W_{\text{up}}, W_{\text{down}}$) | $2 \, d \cdot d_{\text{ff}} + d_{\text{ff}} \cdot d = 3 \, d \cdot d_{\text{ff}}$ |
| Per-block: 2× RMSNorm | $2d$ |
| Final RMSNorm | $d$ |
| LM head | shared — $0$ additional |

: {tbl-colwidths="[40,60]"}

In [ ]:
def count_parameters(config: GPTConfig) -> int:
    d, n, V = config.d_model, config.n_layers, config.vocab_size

    # SwiGLU hidden dim
    d_ff = int(2 * 4 * d / 3)
    d_ff = 64 * ((d_ff + 63) // 64)

    embedding  = V * d
    per_block  = (3*d*d) + (d*d) + (3*d*d_ff) + (2*d)  # attn + ffn + norms
    all_blocks = n * per_block
    final_norm = d

    total = embedding + all_blocks + final_norm
    print(f"Embedding:    {embedding:>12,}  (shared with LM head)")
    print(f"Per block:    {per_block:>12,}  × {n} layers")
    print(f"All blocks:   {all_blocks:>12,}")
    print(f"Final norm:   {final_norm:>12,}")
    print(f"{'─'*38}")
    print(f"Total:        {total:>12,}  ({total/1e6:.1f}M)")
    return total


print("=== GPT-2 small config ===")
count_parameters(GPTConfig())

In [ ]:
@dataclass
class NanoGPTConfig(GPTConfig):
    vocab_size:  int = 50257
    d_model:     int = 384
    n_layers:    int = 6
    n_heads:     int = 6
    max_seq_len: int = 256

print("=== NanoGPT config ===")
count_parameters(NanoGPTConfig())

## Verifying the Implementation

Before training anything, we verify two things: (1) shapes are correct, and (2) the initial loss is in the expected range. At random initialization, cross-entropy loss should be close to $\log(V) \approx \log(50257) \approx 10.82$. [A wildly different initial loss is a strong signal of a bug]{.mark} — too low suggests data leakage or weight initialization issues; too high suggests missing scale factors.

In [ ]:
torch.manual_seed(42)
config = NanoGPTConfig()
model  = GPT(config)

B, T = 4, 64
idx     = torch.randint(0, config.vocab_size, (B, T))
targets = torch.randint(0, config.vocab_size, (B, T))

logits, loss = model(idx, targets)
print(f"logits shape:          {logits.shape}")
print(f"loss:                  {loss.item():.4f}")
print(f"expected (log V):      {math.log(config.vocab_size):.4f}")

In [ ]:
# Gradient flow check — all parameters must receive non-zero gradients
loss.backward()
not_ok = [
    name for name, p in model.named_parameters()
    if p.grad is None or p.grad.norm().item() == 0
]
if not_ok:
    print("WARNING — no gradient:", not_ok)
else:
    print(f"All {sum(p.numel() for p in model.parameters()):,} parameters have gradients.")

## Summary

| Component | Key detail |
|---|---|
| Scaled dot-product attention | $\text{softmax}(QK^\top / \sqrt{d_k})V$ — [scale prevents softmax saturation]{.mark} |
| Causal mask | `torch.triu(..., diagonal=1)` → $-\infty$ before softmax |
| Multi-head attention | $h$ heads, each $d_k = d / h$; merged with $W_O$ |
| RoPE | Rotates Q and K; [dot product depends only on relative offset $m-n$]{.mark} |
| Residual gradient | $\partial \mathcal{L}/\partial x = (\partial \mathcal{L}/\partial y)(I + \partial F/\partial x)$ — identity term is the gradient highway |
| Pre-norm | RMSNorm inside the residual branch; skip-connection gradient path is clean |
| RMSNorm | Normalizes by RMS over feature dim; no mean-centering, no $\beta$ parameter |
| SwiGLU | $(W_1 x \odot \text{SiLU}(W_g x))W_2$ — gated FFN; $d_{\text{ff}} = 8d/3$ rounded to 64 |
| Residual scaling | Output projections initialized with $0.02 / \sqrt{2L}$ — bounds stream variance |
| Weight tying | LM head shares weights with token embedding |
| Sanity check | Initial CE loss $\approx \log(V)$ under correct initialization |

: {tbl-colwidths="[30,70]"}

## Exercises

**1.** Remove the $\sqrt{d_k}$ scale factor from `scaled_dot_product_attention` and train the nano model for 100 steps. Compare the loss curve to the scaled version and explain the difference in terms of the softmax saturation argument.

**2.** Implement `scaled_dot_product_attention` using `torch.nn.functional.scaled_dot_product_attention` (PyTorch's fused kernel, available in PyTorch 2.0+) and verify it produces the same output as the manual implementation.

**3.** Change `TransformerBlock` to use post-norm instead of pre-norm (LayerNorm after the residual addition). Plot the per-layer gradient norm for both variants over 200 training steps and confirm that pre-norm produces more uniform gradient norms across depth.

**4.** Verify the parameter count analytically: iterate over `model.named_parameters()`, sum `param.numel()`, and confirm the total matches `count_parameters`. Identify which component dominates for NanoGPT vs. GPT-2 small.

**5.** Add a `kv_cache` argument to `MultiHeadAttention.forward()` that, when provided, appends the current K and V to the cache and attends over the full cached history. Verify that generation with the cache produces identical token sequences to generation without it.

■